In [ ]:
import os, io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import gzip
import requests

import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import Conv2D, ReLU, MaxPooling2D, UpSampling2D, Dropout, BatchNormalization, Flatten, Dense, Conv2DTranspose, GlobalAveragePooling2D, DepthwiseConv2D
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.makedirs('res', exist_ok=True)
data = requests.get('https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip')
files = zipfile.ZipFile(io.BytesIO(data.content))
files.extractall('res')

validation_split = 0.3
batch_size = 512
num_of_classes = 62 #MNIST ByClass classes

def read_MNIST_images(filename):
    with gzip.open(filename, 'rb') as file:
        images = np.frombuffer(file.read(), np.uint8, offset=16)
    return images.reshape(-1, 28, 28, 1).astype("float32") / 255.0
        
def read_MNIST_labels(filename):
    with gzip.open(filename, 'rb') as file:
        labels = np.frombuffer(file.read(), np.uint8, offset=8)
    return labels

x_train = read_MNIST_images('res/gzip/emnist-byclass-train-images-idx3-ubyte.gz')
y_train = read_MNIST_labels('res/gzip/emnist-byclass-train-labels-idx1-ubyte.gz')
x_test = read_MNIST_images('res/gzip/emnist-byclass-test-images-idx3-ubyte.gz')
y_test = read_MNIST_labels('res/gzip/emnist-byclass-test-labels-idx1-ubyte.gz')

x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=validation_split, random_state=42)

#------------------------------------------------
#Comment this part of code to test the full dataset 
# dataset_sample_val = 0.1
# x_train, _, y_train, _ = train_test_split(x_train, y_train, test_size=(1 - dataset_sample_val), random_state=42)
# x_val, _, y_val, _ = train_test_split(x_val, y_val, test_size=(1 - dataset_sample_val), random_state=42)
# x_test, _, y_test, _ = train_test_split(x_test, y_test, test_size=(1 - dataset_sample_val), random_state=42)
#------------------------------------------------
y_train = to_categorical(y_train, num_of_classes)
y_val = to_categorical(y_val, num_of_classes)
y_test = to_categorical(y_test, num_of_classes)

In [ ]:
val_test_generator = ImageDataGenerator()
train_generator = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.2,
    zoom_range=0.2)

In [ ]:
train_dataset = train_generator.flow(x_train, y_train, batch_size=batch_size)
val_dataset = val_test_generator.flow(x_val, y_val, batch_size=batch_size)
test_dataset = val_test_generator.flow(x_test, y_test, batch_size=batch_size)

In [ ]:
def create_model():
    model = keras.Sequential()
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation=None, padding='same', input_shape=(28, 28, 1)))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(Conv2D(32, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(num_of_classes, activation='softmax')) 

    optimizer = Adam(learning_rate=2e-3)
    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=optimizer,
                  loss=loss_fn,
                  metrics=['accuracy', 'Precision', 'Recall'])
    model.summary()

    return model

model = create_model()

In [ ]:
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, min_lr=2.5e-5)
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model.fit(train_dataset,
         validation_data=val_dataset,
         epochs=30,
         batch_size=batch_size,
         shuffle=True,
         callbacks=[lr_scheduler, callback])

In [ ]:
def get_model_metrics(model, test_data, model_title='model'):
    res = model.evaluate(test_data)
    loss, acc, prec, rec = res
    print(model_title)
    print(f'The accuracy of the model is{acc}')
    print(f'The loss of the model is {loss}')
    print(f'The precision of the model is{prec}')
    print(f'The recall of the model is {rec}')
    print('-' * 150 + '\n')

get_model_metrics(model, test_dataset,model_title='Model metrics for MNIST dataset')

In [ ]:
model.save("models/model_emnist.keras")